In [ ]:
import numpy as np
import pandas as pd

In [ ]:
# 3 utility_functions:
# DONE in preference.py
# 1) Section 3.3 : importance weighted aspect profile with feedback weighting
def aspect_profile(phi_rows: np.ndarray, idx: np.ndarray, alpha: np.ndarray | None = None):
    """
    Compute aspect profile z_t for a selected set S_t (importance weighted).

    Inputs:
      phi_rows: (N, K) full aspect matrix for all rows in your dataset
      idx: list/array of indices that represent the selected set S_t
      alpha: optional weights per selected sentence (importance weighting)

    Output:
      z_t: (K,) vector representing aspect distribution of that selection
    """
    if len(idx) == 0:
        raise ValueError("Empty S_t.")
    if alpha is None:
        return phi_rows[idx].mean(axis=0)
    alpha = np.asarray(alpha, dtype=np.float32)
    alpha = alpha / (alpha.sum() + 1e-12) # sum of alpha_i should be 1 for proper weighting
    return (phi_rows[idx] * alpha[:, None]).sum(axis=0) # weighted average of aspect profiles for selected sentences

# DONE in utils.py
# 2) softmax functions
def softmax(x: np.ndarray, axis=-1):
    """
    softmax turns arbitrary real numbers into a probability distribution. 

    # we would need this to convert aspect scores phi into proper preference vector probabilities.

    Example:
        x = [2, 1, 0]
        softmax(x) -> [0.66, 0.24, 0.09] (roughly)

    Why subtract max?
        Prevents exp(large number) overflow.
    """
    x = np.asarray(x)
    x = x - np.max(x, axis=axis, keepdims=True)  # numerical stability
    ex = np.exp(x)
    return ex / (np.sum(ex, axis=axis, keepdims=True) + 1e-12)

# 3) Get Phi Matrix function
# DONE in extractors/common.py
def get_phi_matrix(df: pd.DataFrame, phi_cols):
    """
    Extract aspect-score columns from df and return a numpy matrix Phi.

    Phi shape: (N, K)
      - N = number of rows (sentences/reviews)
      - K = number of aspects

    We also row-normalize so each row sums to 1 (simplex).
    """
    Phi = df[phi_cols].to_numpy(dtype=np.float32)
    # (optional) row-normalize if you want exact simplex rows # phi_i/sum(phi_i)
    row_sums = Phi.sum(axis=1, keepdims=True)
    Phi = Phi / (row_sums + 1e-12)
    return Phi

In [ ]:
# DONE in preference.py
def estimate_user_pref_Boltzman_from_history(history, phi_rows, beta=5.0, eps=1e-8, prior=None):
    """
    Estimate the user preference vector w_u from interaction history using an empirical Boltzmann estimation.

    history: list of events, each event dict contains:
      {
        "S_idx": np.array([...])   # indices of selected sentences (S_t)
        "f": float                 # feedback in [0,1]
        "alpha": optional weights over S_t
      }

    phi_rows: (N, K) full aspect matrix
    beta: preference sharpness for softmax
    prior: optional prior distribution over aspects (K,)

    Output:
      w_u: (K,) user preference vector in simplex
    """
    K = phi_rows.shape[1]

    # if no history: return prior or uniform
    if (history is None) or (len(history) == 0):
        if prior is None:
            return np.ones(K, dtype=np.float32) / K
        prior = np.asarray(prior, dtype=np.float32)
        return prior / (prior.sum() + 1e-12)

    # numerator and denominator for ztilde_u
    num = np.zeros(K, dtype=np.float32)
    den = 0.0

    # build: ztilde_u = sum_t f_t z_t / sum_t f_t
    # for each event in history, compute aspect profile z_t and accumulate weighted by feedback f_t
    for ev in history:
        f = float(ev["f"])
        S_idx = np.asarray(ev["S_idx"], dtype=int)
        alpha = ev.get("alpha", None)

        z_t = aspect_profile(phi_rows, S_idx, alpha=alpha)  # (K,) aspect profile for this event
        num += f * z_t # accumulate weighted aspect profile
        den += f # accumulate total feedback for normalization

    z_tilde = num / (den + eps)  # (K,) z_tilde_u =(sum_t f_t z_t) / (sum_t f_t + eps) for stability ... this is the feedback-weighted average aspect profile across history

    # Boltzmann / softmax mapping to simplex # section 3.3.1
    w = softmax(beta * z_tilde, axis=0)  

    # optional blending with prior
    if prior is not None:
        prior = np.asarray(prior, dtype=np.float32)
        prior = prior / (prior.sum() + 1e-12)
        w = 0.9 * w + 0.1 * prior
        w = w / (w.sum() + 1e-12)

    return w # output w_u of shape K summing to 1 (simplex)

# here aspects that appear in sets with higher feedback f_t will have higher z_tilde values, and the softmax with beta will amplify the differences to produce a sharper preference vector w_u.

In [ ]:
# DONE in preference.py
# section 3.3 uniform averaging without feedback weighting: w_0
def estimate_user_pref_from_own_text(df_sent, user_id, phi_cols, beta=5.0):
    """
    Bootstraps w_u directly from the user's own sentences/reviews  if present 
    """
    sub = df_sent[df_sent["user_id"] == user_id]
    if len(sub) == 0:
        K = len(phi_cols)
        return np.ones(K, dtype=np.float32) / K
    Phi = get_phi_matrix(sub, phi_cols)      # (m,K)
    ztilde = Phi.mean(axis=0)                # average aspect mass
    return softmax(beta * ztilde, axis=0)

In [ ]:
# DONE in preference.py
# For feedback generation : w^{*} as true user preference vector but hidden (oracle) to generate synthetic feedback
def make_synthetic_true_preference(df_sent, user_id, phi_cols, beta=8.0, noise_scale=0.03, seed=42):
    rng = np.random.default_rng(seed)

    w_boot = estimate_user_pref_from_own_text(
        df_sent=df_sent,
        user_id=user_id,
        phi_cols=phi_cols,
        beta=beta
    )

    noisy_logits = np.log(w_boot + 1e-12) + noise_scale * rng.standard_normal(len(w_boot))
    w_true = softmax(noisy_logits, axis=0)
    return w_true

In [ ]:
# DONE in utils.py
def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

# DONE in feedback.py
def synthetic_feedback_from_profile(
    z_t,
    w_true,
    gamma=10.0,
    noise_std=0.05,
    seed=None
):
    """
    Generate synthetic scalar feedback in [0,1] from aspect profile z_t.
    Larger if z_t aligns well with hidden true preference w_true.
    """
    rng = np.random.default_rng(seed)
    score = float(np.dot(w_true, z_t))
    noisy_score = score + rng.normal(0.0, noise_std)

    # map to [0,1]
    f_t = sigmoid(gamma * (noisy_score - np.mean(w_true)))
    return float(np.clip(f_t, 0.0, 1.0))

In [ ]:
# DONE in extractors/common.py
def ensure_length_col(df: pd.DataFrame, text_col="review_text", length_col="len"):
    """
    Adds a column df[length_col] = length of each sentence string.

    Why?
      - We enforce length budget L in extraction
      - We optionally penalize too-long sentences

    If already present, does nothing.
    """
    if length_col not in df.columns:
        df = df.copy()
        df[length_col] = df[text_col].fillna("").astype(str).str.len().astype(int)
    return df

# DONE in extractors/gumbel.py
def sample_gumbel(shape, rng=None):
    """
    Generate Gumbel(0,1) noise. Section 3.1.2  describes g_{t',j}

    If U ~ Uniform(0,1), then:
      G = -log(-log(U)) is ensures Gumbel distribution
    """
    rng = np.random.default_rng() if rng is None else rng
    U = rng.uniform(low=1e-12, high=1.0 - 1e-12, size=shape)
    return -np.log(-np.log(U))

# DONE in extractors/gumbel.py
def gumbel_priority_order(U, tau_ext=1.0, seed=0):
    """
    Create a random ordering of sentences where higher-utility sentences
    tend to appear earlier.

    # section 3.1.2 describes this process of creating a random priority ordering using gumbel-perturbed utilities.
    Steps:
      1) sample g_i ~ Gumbel
      2) xi_i = tau_ext * U_i + g_i
      3) sort by xi in descending order

    Output:
      perm: indices sorted high-to-low
      xi: perturbed scores
    """
    rng = np.random.default_rng(seed)
    g = sample_gumbel(len(U), rng=rng)
    xi = tau_ext * np.asarray(U, dtype=np.float32) + g
    perm = np.argsort(-xi)  # descending
    return perm, xi

# DONE in extractors/gumbel.py
def gumbel_priority_greedy_select(df_candidates, U, k=8, L=800, len_col="len", tau_ext=1.0, seed=0):
    """
    Implements algorithm 2 : section 3.1.2

    - create random priority ordering using gumbel-perturbed utilities
    - scan in that order
    - add sentence if it doesn't violate constraints:
        (1) |S| <= k
        (2) total length <= L

    Output:
      selected_local_idx: indices (within df_candidates) that were selected
      perm: the full ordering
      xi: reminder scores
      total_len: length used
    """
    perm, xi = gumbel_priority_order(U, tau_ext=tau_ext, seed=seed)

    selected = []
    total_len = 0
    lens = df_candidates[len_col].to_numpy(dtype=int)

    for j in perm:
        if len(selected) >= k:
            break
        if total_len + int(lens[j]) <= L:
            selected.append(j)
            total_len += int(lens[j])

    return np.array(selected, dtype=int), perm, xi, total_len

# this is section 3.1
# DONE in extractors/common.py
def compute_sentence_utilities(df_candidates, phi_cols, w_u, lam=0.0, len_col="len"):
    """
    For a product p, we have candidate sentences.

    Compute for each sentence i:
      base_i = w_u^T phi_i   (how much it matches the user's aspects)

    Optionally penalize long sentences:
      base_i -= lam * len_i / mean_len

    This is the relevance score U_i for each candidate sentence.
    """
    Phi = get_phi_matrix(df_candidates, phi_cols)         # (n_p, K)
    w_u = np.asarray(w_u, dtype=np.float32)               # (K,)

    # aspect relevance score per sentence
    U = Phi @ w_u                                         # (n_p,)

    # optional length penalty
    if lam > 0:
        lens = df_candidates[len_col].to_numpy(dtype=np.float32)
        mean_len = float(np.mean(lens) + 1e-12)
        U = U - lam * (lens / mean_len)

    return U, Phi


# DONE in extractors/base.py extract
def extract_for_user_product(
    df_sent: pd.DataFrame,
    user_id: str,
    product_id: str,
    phi_cols,
    w_u: np.ndarray,
    k=8,
    L=800,
    lam=0.0,
    tau_ext=1.0,
    tau_alpha=1.0,
    alpha_mode="utility_softmax",   # "uniform", "utility_softmax", "rank_softmax", "blended"
    rank_decay=0.3,
    seed=0,
    text_col="review_text",
    len_col="len",
):
    """
    Main function to call for algorithm 2 implementation : Section 3.1.2

    Outputs:
      selected_df: extracted rows
      alpha: importance weights over selected rows
      z_t: weighted aspect profile of extracted set
    """
    # DONE in extractors/common.py
    df_sent = ensure_length_col(df_sent, text_col=text_col, length_col=len_col)

    cand = df_sent[df_sent["parent_asin"] == product_id].copy()
    if len(cand) == 0:
        return None
    

    U, Phi = compute_sentence_utilities(cand, phi_cols, w_u, lam=lam, len_col=len_col)


    # DONE in extractors/gumbel.py: _select_local_indices
    sel_local_idx, perm, xi, total_len = gumbel_priority_greedy_select(
        cand, U, k=k, L=L, len_col=len_col, tau_ext=tau_ext, seed=seed
    )

    if len(sel_local_idx) == 0:
        return None

    # selected rows in extraction order
    # DONE in extractors/gumbel.py: _finalize_selected_df
    selected_df = cand.iloc[sel_local_idx].copy()
    selected_df["utility"] = U[sel_local_idx]
    selected_df["gumbel_score"] = xi[sel_local_idx]
    # sort for user-facing priority
    selected_df = selected_df.sort_values("gumbel_score", ascending=False).copy()

    # IMPORTANT:
    # match U_sel and Phi_sel to the same final order as selected_df
    final_idx = selected_df.index.to_numpy()
    local_pos = cand.index.get_indexer(final_idx)

    U_sel = U[local_pos]
    Phi_sel = Phi[local_pos]

    # choose alpha: DONE in extractors/common.py
    if alpha_mode == "uniform":
        alpha = np.ones(len(U_sel), dtype=np.float32) / len(U_sel)

    elif alpha_mode == "utility_softmax":
        alpha = softmax(tau_alpha * U_sel, axis=0)

    elif alpha_mode == "rank_softmax":
        ranks = np.arange(len(U_sel), dtype=np.float32)
        alpha = softmax(-rank_decay * ranks, axis=0)

    elif alpha_mode == "blended":
        ranks = np.arange(len(U_sel), dtype=np.float32)
        scores = tau_alpha * U_sel - rank_decay * ranks
        alpha = softmax(scores, axis=0)

    else:
        raise ValueError(f"Unknown alpha_mode: {alpha_mode}")
    
    z_t = aspect_profile(phi_rows=Phi_sel, idx=final_idx, alpha=alpha)
    # z_t = (Phi_sel * alpha[:, None]).sum(axis=0)

    selected_df["alpha"] = alpha

    return {
        "selected_df": selected_df,
        "alpha": alpha,
        "z_t": z_t,
        "total_len": total_len,
        "n_candidates": len(cand),
        "alpha_mode": alpha_mode,
    }
        

In [ ]:
def build_history_event(selected_df, z_t, feedback, product_id, user_id, alpha=None, method="gumbel"):
    return {
        "user_id": user_id,
        "product_id": product_id,
        "S_idx": selected_df["global_idx"].to_numpy(dtype=int),
        "z_t": np.asarray(z_t, dtype=np.float32),
        "f": float(feedback),
        "alpha": alpha,
        "method": method,
    }

In [ ]:
def attach_global_index(df):
    df = df.copy()
    df["global_idx"] = np.arange(len(df))
    return df

data = pd.read_csv("../data/train_with_aspect_scores.csv")
df_sent = attach_global_index(data)

K = 20
phi_cols = [f"asp_{k}" for k in range(K)]
# 1) bootstrap user preference from user's own text (or use history-based)
w_u = estimate_user_pref_from_own_text(data, user_id="AG73BVBKUOH22USSFJA5ZWL7AKXA", phi_cols=phi_cols, beta=8.0)

# 2) extract sentences for a product using gumbel-priority greedy
out = extract_for_user_product(
    df_sent=data,
    user_id="AG73BVBKUOH22USSFJA5ZWL7AKXA",
    product_id="B085BB7B1M",
    phi_cols=phi_cols,
    w_u=w_u,
    k=10,
    L=900,
    lam=0.2,
    tau_ext=2.0,
    seed=42,
)

selected_df = out["selected_df"]
alpha = out.get("alpha", None)

In [ ]:
def init_boltz_state(K):
    return {
        "num": np.zeros(K, dtype=np.float32),   # sum_t f_t z_t
        "den": 0.0,                             # sum_t f_t
    }


def online_boltz_update(state, z_t, f_t, beta=5.0, eps=1e-8, prior=None):
    """
    One-step online empirical Boltzmann update.
    """
    state["num"] += float(f_t) * np.asarray(z_t, dtype=np.float32)
    state["den"] += float(f_t)

    z_tilde = state["num"] / (state["den"] + eps)
    w = softmax(beta * z_tilde, axis=0)

    if prior is not None:
        prior = np.asarray(prior, dtype=np.float32)
        prior = prior / (prior.sum() + 1e-12)
        w = 0.9 * w + 0.1 * prior
        w = w / (w.sum() + 1e-12)

    return w, state

In [ ]:
def init_omd_state(w_init):
    w_init = np.asarray(w_init, dtype=np.float32)
    w_init = w_init / (w_init.sum() + 1e-12)
    return {
        "w": w_init.copy()
    }


def online_omd_update(state, z_t, f_t, eta=1.0):
    """
    Entropic OMD / exponentiated gradient update:
        w_{t+1,k} ∝ w_{t,k} exp(eta * f_t * z_{t,k})
    """
    w_t = np.asarray(state["w"], dtype=np.float32)
    z_t = np.asarray(z_t, dtype=np.float32)
    mult = np.exp(eta * float(f_t) * z_t)

    w_next = w_t * mult
    w_next = w_next / (w_next.sum() + 1e-12)

    state["w"] = w_next
    return w_next, state

In [ ]:
def simulate_online_learning_gumbel(
    df_sent,
    user_id,
    product_ids,
    phi_cols,
    beta_init=8.0,
    beta_boltz=8.0,
    eta_omd=3.0,
    k=10,
    L=900,
    lam=0.2,
    tau_ext=2.0,
    seed=42
):
    rng = np.random.default_rng(seed)

    df_sent = attach_global_index(df_sent)
    K = len(phi_cols)
    phi_rows = get_phi_matrix(df_sent, phi_cols)

    # initial estimate from user's own text
    w_init = estimate_user_pref_from_own_text(
        df_sent=df_sent,
        user_id=user_id,
        phi_cols=phi_cols,
        beta=beta_init
    )

    # hidden oracle for synthetic feedback
    w_true = make_synthetic_true_preference(
        df_sent=df_sent,
        user_id=user_id,
        phi_cols=phi_cols,
        beta=beta_init,
        noise_scale=0.03,
        seed=seed
    )

    boltz_state = init_boltz_state(K)
    omd_state = init_omd_state(w_init)

    history = []
    logs = []

    w_boltz = w_init.copy()
    w_omd = w_init.copy()

    for t, product_id in enumerate(product_ids):
        # choose one of the learners to drive extraction
        # here I use OMD weights as the current online policy
        w_current = w_omd

        out = extract_for_user_product(
            df_sent=df_sent,
            user_id=user_id,
            product_id=product_id,
            phi_cols=phi_cols,
            w_u=w_current,
            k=k,
            L=L,
            lam=lam,
            tau_ext=tau_ext,
            seed=seed + t,
        )

        if out is None or len(out["selected_df"]) == 0:
            continue

        selected_df = out["selected_df"]
        z_t = out["z_t"]
        alpha = out["alpha"]

        # synthetic feedback
        f_t = synthetic_feedback_from_profile(
            z_t=z_t,
            w_true=w_true,
            gamma=12.0,
            noise_std=0.03,
            seed=seed + 1000 + t
        )

        event = build_history_event(
            selected_df=selected_df,
            z_t=z_t,
            feedback=f_t,
            product_id=product_id,
            user_id=user_id,
            alpha=alpha,
            method="gumbel",
        )
        history.append(event)

        # update Boltzmann
        w_boltz, boltz_state = online_boltz_update(
            boltz_state,
            z_t=z_t,
            f_t=f_t,
            beta=beta_boltz,
            prior=w_init
        )

        # update OMD
        w_omd, omd_state = online_omd_update(
            omd_state,
            z_t=z_t,
            f_t=f_t,
            eta=eta_omd
        )

        logs.append({
            "t": t,
            "product_id": product_id,
            "feedback": f_t,
            "z_t": z_t.copy(),
            "w_boltz": w_boltz.copy(),
            "w_omd": w_omd.copy(),
            "w_true": w_true.copy(),
        })

    return {
        "history": history,
        "logs": logs,
        "w_init": w_init,
        "w_true": w_true,
        "w_boltz_final": w_boltz,
        "w_omd_final": w_omd,
    }

In [ ]:
all_products = data["parent_asin"].dropna().unique().tolist()

result = simulate_online_learning_gumbel(
    df_sent=data,
    user_id="AG73BVBKUOH22USSFJA5ZWL7AKXA",
    product_ids=all_products[:20],   # simulate first 20 interactions
    phi_cols=phi_cols,
    beta_init=8.0,
    beta_boltz=8.0,
    eta_omd=3.0,
    k=10,
    L=900,
    lam=0.2,
    tau_ext=2.0,
    seed=42
)

print("Initial w:\n", result["w_init"])
print("\nTrue hidden w*:\n", result["w_true"])
print("\nFinal Boltzmann w:\n", result["w_boltz_final"])
print("\nFinal OMD w:\n", result["w_omd_final"])
print("\nNumber of events:", len(result["history"]))